In [1]:
whole_smiles = "[2H]C([2H])([2H])NC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4"
scaffold_smiles ="COc1ccccc1Nc1cc(NC(=O)C2CC2)nnc1C=O"
no_mols = 10000
sim_mode = "ecfp4"
output_dir = f"Deucravacitinib/rgrp_enum/{sim_mode}"

**Load Fragment Library**

In [2]:
from molshift.tools.frag_lib_loader import load_fragment_data


fragment_lib = load_fragment_data()['fragment_desc']

In [3]:
from pathlib import Path
from molshift.tools.rgrp_enum import run_rgrp

rgrp_enum_data = run_rgrp(
    whole_smiles = whole_smiles,
    scaffold_smiles = scaffold_smiles,
    no_rgrp_mols = no_mols,
    fragment_desc = fragment_lib,
    output_dir = Path(output_dir)/"rgrp_enum/desc",
    sim_mode = sim_mode)


Total R-groups: 2, Cutoff per site: 100, Total mols: 10000
att=21: ✓ saved 100 fragments → Deucravacitinib\rgrp_enum\ecfp4\rgrp_enum\desc\Similar_fragments\att_21.csv
att=3: ✓ saved 100 fragments → Deucravacitinib\rgrp_enum\ecfp4\rgrp_enum\desc\Similar_fragments\att_3.csv

 Total Generated mols:  10000
Output saved to: Deucravacitinib\rgrp_enum\ecfp4\rgrp_enum\desc/rgrp_all.csv


In [4]:
import pandas as pd

rgrp_enum_df = pd.read_csv(rgrp_enum_data['Output_path'])
selected_atts = rgrp_enum_data['selected_attachments']

In [5]:
import json, sys, platform, datetime, pathlib

def save_manifest(params: dict, path=f"{output_dir}/manifest.json"):
    manifest = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "params": params,
    }
    pathlib.Path(path).write_text(json.dumps(manifest, indent=2, default=str))
    return manifest

# Example
params = {
    "Whole smiles": whole_smiles,
    "Scaffold smiles": scaffold_smiles,
    "Similarity screening mode": sim_mode,
    "Number of rgrp replaced": len(selected_atts),
    "Rgrp enumeration mols requested": no_mols,
    "Number of rgrp_enum": len(rgrp_enum_df),
}
save_manifest(params)

{'timestamp': '2026-11-15T17:48:05',
 'python': '3.14.2',
 'platform': 'Windows-11-10.0.26200-SP0',
 'params': {'Whole smiles': '[2H]C([2H])([2H])NC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4',
  'Scaffold smiles': 'COc1ccccc1Nc1cc(NC(=O)C2CC2)nnc1C=O',
  'Similarity screening mode': 'ecfp4',
  'Number of rgrp replaced': 2,
  'Rgrp enumeration mols requested': 10000,
  'Number of rgrp_enum': 10000}}

**Filteration**

In [6]:
from molshift.tools.ref_filter import filter_by_reference_parallel
import os

filtered, windows = filter_by_reference_parallel(rgrp_enum_df, "smiles", whole_smiles,n_jobs = os.cpu_count()-4,
                                      sa_buffer=1.0, qed_buffer=0.1)
filtered.to_csv(Path(output_dir)/"rgrp_enumerated_filtered.csv", index=False)

Filtering (12 procs): 100%|██████████| 2/2 [00:29<00:00, 14.58s/it]


**Docking**

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.WARNING)
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")
from vina_docking.tools.ligand_docking import run_ligand_docking


dock_config_path = r"dock_config.json"
PROT_DIR = r"Docking_Validations\TYK2"
BASE_NAME = "6NZP"
OUT_DIR = Path(output_dir)/"Docking"
LIGAND_FILE = Path(output_dir)/"rgrp_enumerated_filtered.csv"
SMILES_COL = "smiles"
K = 30
HAS_REFERENCE = True
SCORE_THRESHOLD = 0.5
REF_PATH = r"Docking_Validations\TYK2\REFERENCE"
run_ligand_docking(
        PROT_DIR,
        BASE_NAME,
        OUT_DIR,
        LIGAND_FILE,
        SMILES_COL,
        K,
        HAS_REFERENCE,
        SCORE_THRESHOLD,
        REF_PATH,
        dock_config_path,
    )